# Support Ticket Volume Analysis

## Project Overview
Analyze support ticket volume over time, by category, severity, channel, and customer segment. Identify major workload drivers, peak periods, and operational insights for capacity planning and service improvement.

## Learning Objectives
- Analyze support ticket trends across multiple dimensions
- Identify seasonal and day-of-week patterns in ticket volume
- Segment ticket drivers by category, severity, and customer type
- Generate capacity planning and process improvement recommendations

## Problem Statement
A support team is overwhelmed during certain periods and wants to understand ticket volume patterns to staff appropriately, reduce common ticket types, and improve response times.

## Why This Project Matters
Support costs scale with ticket volume. Understanding volume patterns enables proactive staffing, self-service deflection for common issues, and root cause fixes that reduce ticket inflow permanently.

## Dataset Overview
Synthetic support ticket data: ~12K tickets over 2 years with category, severity, channel, customer segment, resolution time, and satisfaction score.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by typical SaaS/e-commerce support operations
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_tickets = 12000

categories = ['Billing', 'Technical Issue', 'Account Access', 'Feature Request',
              'Bug Report', 'How-To Question', 'Cancellation']
cat_weights = [0.18, 0.22, 0.12, 0.10, 0.15, 0.13, 0.10]

severities = ['Low', 'Medium', 'High', 'Critical']
sev_weights = [0.30, 0.35, 0.25, 0.10]

channels = ['Email', 'Chat', 'Phone', 'Self-Service Portal']
ch_weights = [0.30, 0.30, 0.20, 0.20]

segments = ['Free', 'Pro', 'Enterprise']
seg_weights = [0.40, 0.40, 0.20]

# Generate dates with seasonality (more tickets in Jan, after holidays, and Mondays)
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
date_weights = np.ones(len(dates))
for i, d in enumerate(dates):
    if d.month in [1, 7]: date_weights[i] *= 1.3  # post-holiday & mid-year spikes
    if d.dayofweek == 0: date_weights[i] *= 1.25  # Monday spike
    if d.dayofweek >= 5: date_weights[i] *= 0.4   # weekend dip
date_weights /= date_weights.sum()

rows = []
for i in range(n_tickets):
    cat = np.random.choice(categories, p=cat_weights)
    sev = np.random.choice(severities, p=sev_weights)
    channel = np.random.choice(channels, p=ch_weights)
    segment = np.random.choice(segments, p=seg_weights)
    date = np.random.choice(dates, p=date_weights)

    # Resolution time varies by severity and channel
    base_hours = {'Low': 4, 'Medium': 12, 'High': 24, 'Critical': 48}
    res_hours = max(0.5, np.random.exponential(scale=base_hours[sev]))
    res_hours = round(res_hours, 1)

    # CSAT (1-5) inversely related to resolution time
    csat = np.clip(np.random.normal(4.0 - res_hours/50, 0.8), 1, 5)

    rows.append({
        'TicketID': f'TKT{i:06d}', 'CreatedDate': date, 'Category': cat,
        'Severity': sev, 'Channel': channel, 'CustomerSegment': segment,
        'ResolutionHours': res_hours, 'CSAT': round(csat, 1),
    })

df = pd.DataFrame(rows)
df['CreatedDate'] = pd.to_datetime(df['CreatedDate'])
df['YearMonth'] = df['CreatedDate'].dt.to_period('M')
df['DayOfWeek'] = df['CreatedDate'].dt.day_name()
df['Month'] = df['CreatedDate'].dt.month
print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["CreatedDate"].min().date()} to {df["CreatedDate"].max().date()}')
print(f'Avg resolution hours: {df["ResolutionHours"].mean():.1f}')
print(f'Avg CSAT: {df["CSAT"].mean():.2f}')
print(f'\nCategory distribution:\n{df["Category"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Monthly ticket volume
monthly = df.groupby('YearMonth').size()
monthly.plot(ax=axes[0,0], marker='o')
axes[0,0].set_title('Monthly Ticket Volume')
axes[0,0].tick_params(axis='x', rotation=45)

# By category
df['Category'].value_counts().plot.barh(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Tickets by Category')
axes[0,1].invert_yaxis()

# By severity
sev_order = ['Low', 'Medium', 'High', 'Critical']
df['Severity'].value_counts().reindex(sev_order).plot.bar(ax=axes[1,0], edgecolor='black',
    color=['#66bb6a','#ffa726','#ef5350','#b71c1c'])
axes[1,0].set_title('Tickets by Severity')
axes[1,0].tick_params(axis='x', rotation=0)

# By channel
df['Channel'].value_counts().plot.bar(ax=axes[1,1], edgecolor='black', color='mediumpurple')
axes[1,1].set_title('Tickets by Channel')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Day-of-Week Volume Pattern

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts = df['DayOfWeek'].value_counts().reindex(dow_order)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ef5350' if d in ['Monday'] else '#ffa726' if d in ['Saturday','Sunday'] else '#66bb6a' for d in dow_order]
dow_counts.plot.bar(ax=ax, edgecolor='black', color=colors)
ax.set_title('Ticket Volume by Day of Week')
ax.set_ylabel('Tickets')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'day_of_week.png'), dpi=100, bbox_inches='tight')
plt.show()
print(dow_counts)

## Resolution Time Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Resolution by severity
sns.boxplot(data=df, x='Severity', y='ResolutionHours', order=sev_order, ax=axes[0],
            palette=['#66bb6a','#ffa726','#ef5350','#b71c1c'])
axes[0].set_title('Resolution Time by Severity')
axes[0].set_ylabel('Hours')

# Resolution by category
cat_res = df.groupby('Category')['ResolutionHours'].median().sort_values(ascending=False)
cat_res.plot.barh(ax=axes[1], edgecolor='black', color='coral')
axes[1].set_title('Median Resolution Time by Category')
axes[1].set_xlabel('Hours')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'resolution_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Category × Severity Heatmap

In [ ]:
cat_sev = pd.crosstab(df['Category'], df['Severity'])[sev_order]
plt.figure(figsize=(10, 6))
sns.heatmap(cat_sev, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Ticket Count: Category × Severity')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'category_severity.png'), dpi=100, bbox_inches='tight')
plt.show()

## Customer Segment Analysis

In [ ]:
seg_stats = df.groupby('CustomerSegment').agg(
    tickets=('TicketID', 'count'),
    avg_resolution=('ResolutionHours', 'mean'),
    avg_csat=('CSAT', 'mean'),
    pct_critical=('Severity', lambda x: (x == 'Critical').mean()),
).round(3)
seg_stats['tickets_per_100'] = (seg_stats['tickets'] / seg_stats['tickets'].sum() * 100).round(1)
print('Customer Segment Analysis:')
print(seg_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
seg_stats['avg_csat'].plot.bar(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Avg CSAT by Customer Segment')
axes[0].set_ylabel('CSAT (1-5)')
axes[0].tick_params(axis='x', rotation=0)

seg_stats['avg_resolution'].plot.bar(ax=axes[1], edgecolor='black', color='coral')
axes[1].set_title('Avg Resolution Hours by Segment')
axes[1].set_ylabel('Hours')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'segment_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Channel Efficiency

In [ ]:
channel_stats = df.groupby('Channel').agg(
    tickets=('TicketID', 'count'),
    avg_resolution=('ResolutionHours', 'mean'),
    avg_csat=('CSAT', 'mean'),
    pct_high_sev=('Severity', lambda x: (x.isin(['High','Critical'])).mean()),
).round(3).sort_values('avg_resolution')
print('Channel Efficiency:')
print(channel_stats)

## Workload Drivers Summary

In [ ]:
print('=' * 60)
print('SUPPORT WORKLOAD DRIVERS')
print('=' * 60)
print(f'\nTotal tickets (2 years): {len(df):,}')
print(f'Avg monthly volume: {len(df)/24:.0f}')
print(f'\nTop 3 categories by volume:')
for cat, count in df['Category'].value_counts().head(3).items():
    print(f'  {cat}: {count} ({count/len(df):.0%})')
print(f'\nPeak day: Monday ({dow_counts["Monday"]:,} tickets)')
print(f'Peak months: January and July (post-holiday / mid-year spikes)')
print(f'\nCritical tickets: {(df["Severity"]=="Critical").sum()} ({(df["Severity"]=="Critical").mean():.1%})')
print(f'Avg resolution (Critical): {df[df["Severity"]=="Critical"]["ResolutionHours"].mean():.1f} hours')
print(f'\nRecommendations:')
print('  1. Staff up on Mondays and post-holiday periods')
print('  2. Build self-service content for How-To Questions (13% of volume)')
print('  3. Investigate Technical Issue root causes (22% of volume)')
print('  4. Prioritize billing process improvements to reduce billing tickets')

## Interpretation and Business Insight
- **Technical Issues** and **Billing** are the top two ticket drivers — together they account for ~40% of volume
- **Monday** spikes suggest customers accumulate issues over the weekend and report them on Monday
- **January** and **July** spikes align with post-holiday returns/billing issues and mid-year account reviews
- **Critical** tickets average much longer resolution times, straining the team disproportionately
- **Self-Service Portal** shows comparable CSAT to human channels for certain categories, suggesting deflection potential

## Limitations
- Synthetic data with pre-set patterns — real ticket data has more complex seasonality
- No agent-level analysis (workload distribution, individual performance)
- CSAT is simulated; real CSAT depends on many factors beyond resolution time
- No text analysis of ticket content for deeper categorization

## How to Improve This Project
- Use real helpdesk data (Zendesk/Freshdesk exports) for more realistic analysis
- Add agent productivity and utilization analysis
- Build a ticket volume forecasting model for capacity planning
- Apply text classification to auto-categorize tickets
- Analyze first-contact resolution rate and its impact on CSAT

## Production Considerations
- Set up real-time ticket volume dashboards with alerts for volume spikes
- Implement automated ticket routing based on category and severity
- Track SLA compliance (% resolved within target time by severity)
- Build knowledge base articles for top self-serviceable categories

## Common Mistakes
- Treating all tickets equally regardless of severity and customer segment
- Optimizing for resolution speed without monitoring CSAT
- Staffing based on average volume instead of peak patterns
- Not analyzing ticket deflection opportunities (self-service, chatbots)

## Mini Challenge / Exercises
1. What percentage of tickets could potentially be deflected to self-service? (Hint: How-To + Account Access)
2. Calculate the SLA compliance rate if Critical < 24h, High < 48h, Medium < 72h, Low < 96h.
3. Which channel has the best CSAT-to-resolution-time ratio?
4. If Technical Issue volume could be reduced by 30%, how many fewer tickets per month?

## Final Summary / Key Takeaways
- Support ticket analysis reveals volume patterns critical for staffing and capacity planning
- Category and severity breakdowns identify the biggest workload drivers
- Day-of-week and seasonal patterns enable proactive resource allocation
- Self-service deflection of simple categories is the highest-leverage cost reduction strategy
- Resolution time and CSAT must be balanced — fast but unsatisfying resolutions create repeat tickets